<a href="https://colab.research.google.com/github/novalrnv/244107020210-mobile-course/blob/main/JS09/JS09-Tugas1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# JS09 - Approximate Nearest Neighbors (ANN)
## Tugas Lab 1

**Soal:** Lakukan percobaan pada metric distance yang berbeda, 1000 vs 1.000.000 data, 2D vs 5D data, untuk algoritma ANNOY, FAISS, dan HNSW. Catat performansinya dalam bentuk tabel.

**Rancangan percobaan:**
- **Metric distance:** Euclidean dan Cosine (metric yang didukung ketiga library).
- **Jumlah data / dimensi:** 1000/2D, 1000/5D, 1.000.000/2D, 1.000.000/5D.
- **Algoritma:** ANNOY (10 tree), FAISS (IVFFlat), HNSW (hnswlib).
- **Yang dicatat:** waktu build, waktu query (ms/query), dan recall@10 terhadap exact NN (brute force).

> **Perhatian:** percobaan 1.000.000 data cukup lama (build HNSW di modul sampai sekitar 157 detik per konfigurasi, dan ada 4 konfigurasi besar). Kalau Colab ngadat atau memori habis, turunkan `N_LARGE` jadi `100_000`.

## Langkah 1 - Install dan Import Library

In [1]:
!pip install annoy faiss-cpu hnswlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.5/647.5 kB 13.6 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 32.5 MB/s  0:00:00
  Created wheel for annoy: filename=annoy-1.17.3-cp313-cp313-linux_x86_64.whl size=520957 sha256=d90bceb451e88f76733a8aa7609895e7eee0833fcbf32eb83fb3920d22bb186a
  Stored in directory: /root/.cache/pip/wheels/fc/4e/87/f1f957e7382aa370452cff98276f51686924d5415e449800ed
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679070 sha256=ef6ec67ea1c283ba57fc74f90846b1f75d09ba70aceae4af3c6e6560e2c01577
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f5

In [2]:
import time
import numpy as np
import pandas as pd
import faiss
import hnswlib
from annoy import AnnoyIndex

## Langkah 2 - Fungsi Percobaan

Satu fungsi untuk menjalankan ketiga algoritma pada satu konfigurasi (jumlah data, dimensi, metric). Catatan:
- **Cosine** pada FAISS dilakukan dengan menormalisasi vektor lalu memakai inner product. ANNOY memakai `angular`, HNSW memakai `cosine`.
- **Recall@10** = berapa persen dari 10 tetangga terdekat versi exact yang berhasil ditemukan.
- Waktu query dihitung rata-rata per query (satu query per pemanggilan, agar adil untuk ketiganya).

In [3]:
def run_experiment(n, dim, metric, nq=100, k=10, seed=42):
    rng = np.random.RandomState(seed)
    X = rng.random_sample((n, dim)).astype('float32')
    Q = rng.random_sample((nq, dim)).astype('float32')
    euclid = (metric == 'euclidean')

    # Versi data untuk FAISS (cosine -> normalisasi + inner product)
    if euclid:
        Xf, Qf = X, Q
        gt_index = faiss.IndexFlatL2(dim)
    else:
        Xf, Qf = X.copy(), Q.copy()
        faiss.normalize_L2(Xf)
        faiss.normalize_L2(Qf)
        gt_index = faiss.IndexFlatIP(dim)

    # Ground truth (exact NN)
    gt_index.add(Xf)
    _, gt = gt_index.search(Qf, k)

    def recall(I):
        return np.mean([len(set(I[i]) & set(gt[i])) / k for i in range(nq)])

    res = {}

    # ---------- ANNOY ----------
    t0 = time.time()
    ann = AnnoyIndex(dim, 'euclidean' if euclid else 'angular')
    for i in range(n):
        ann.add_item(i, X[i])
    ann.build(10)  # 10 trees
    build = time.time() - t0

    t0 = time.time()
    I = [ann.get_nns_by_vector(q, k) for q in Q]
    qtime = (time.time() - t0) * 1000 / nq
    res['ANNOY'] = (build, qtime, recall(I))

    # ---------- FAISS (IVFFlat) ----------
    nlist = max(4, int(np.sqrt(n) / 2))
    quantizer = faiss.IndexFlatL2(dim) if euclid else faiss.IndexFlatIP(dim)
    mt = faiss.METRIC_L2 if euclid else faiss.METRIC_INNER_PRODUCT

    t0 = time.time()
    fidx = faiss.IndexIVFFlat(quantizer, dim, nlist, mt)
    fidx.train(Xf)
    fidx.add(Xf)
    build = time.time() - t0
    fidx.nprobe = min(10, nlist)

    t0 = time.time()
    I = [fidx.search(q.reshape(1, -1), k)[1][0] for q in Qf]
    qtime = (time.time() - t0) * 1000 / nq
    res['FAISS'] = (build, qtime, recall(I))

    # ---------- HNSW ----------
    t0 = time.time()
    h = hnswlib.Index(space='l2' if euclid else 'cosine', dim=dim)
    h.init_index(max_elements=n, ef_construction=200, M=16)
    h.add_items(X)
    build = time.time() - t0
    h.set_ef(50)

    t0 = time.time()
    I = [h.knn_query(q.reshape(1, -1), k=k)[0][0] for q in Q]
    qtime = (time.time() - t0) * 1000 / nq
    res['HNSW'] = (build, qtime, recall(I))

    return res

## Langkah 3 - Jalankan Semua Konfigurasi

In [4]:
N_LARGE = 1_000_000   # turunkan ke 100_000 jika memori / waktu terbatas

configs = [(1000, 2), (1000, 5), (N_LARGE, 2), (N_LARGE, 5)]
metrics = ['euclidean', 'cosine']

rows = []
for n, dim in configs:
    for metric in metrics:
        print(f'Running: n={n}, dim={dim}D, metric={metric} ...')
        res = run_experiment(n, dim, metric)
        for algo, (build, qtime, rec) in res.items():
            rows.append({
                'Jumlah Data/Dimensi': f'{n}/{dim}D',
                'Metric': metric,
                'Algoritma': algo,
                'Build (s)': build,
                'Query (ms/query)': qtime,
                'Recall@10': rec,
            })

df_res = pd.DataFrame(rows)
df_res.to_csv('hasil_tugas_lab1.csv', index=False)
print('Selesai.')

Running: n=1000, dim=2D, metric=euclidean ...
Running: n=1000, dim=2D, metric=cosine ...
Running: n=1000, dim=5D, metric=euclidean ...
Running: n=1000, dim=5D, metric=cosine ...
Running: n=1000000, dim=2D, metric=euclidean ...
Running: n=1000000, dim=2D, metric=cosine ...
Running: n=1000000, dim=5D, metric=euclidean ...
Running: n=1000000, dim=5D, metric=cosine ...
Selesai.


## Langkah 4 - Tabel Hasil

Tabel dibuat per metrik pengukuran dengan format seperti pada soal (baris = jumlah data/dimensi + metric, kolom = ANNOY, FAISS, HNSW).

In [5]:
order_idx = ['Jumlah Data/Dimensi', 'Metric']

def make_table(value_col):
    t = df_res.pivot_table(index=order_idx, columns='Algoritma', values=value_col)
    t = t[['ANNOY', 'FAISS', 'HNSW']]
    # pertahankan urutan konfigurasi sesuai percobaan
    order = [(f'{n}/{d}D', m) for n, d in configs for m in metrics]
    return t.loc[order]

print('=== Waktu Build (detik) ===')
display(make_table('Build (s)').round(4))

=== Waktu Build (detik) ===


Algoritma                        ANNOY   FAISS      HNSW
Jumlah Data/Dimensi Metric                              
1000/2D             euclidean   0.0145  0.0010    0.0281
                    cosine      0.0197  0.0010    0.0222
1000/5D             euclidean   0.0108  0.0009    0.0326
                    cosine      0.0139  0.0010    0.0336
1000000/2D          euclidean  14.0031  0.2093   68.0425
                    cosine     40.6867  0.4781   96.9442
1000000/5D          euclidean  10.7854  0.3235  110.2205
                    cosine     14.0388  0.9096   99.4891

In [6]:
print('=== Waktu Query (ms/query) ===')
display(make_table('Query (ms/query)').round(4))

=== Waktu Query (ms/query) ===


Algoritma                       ANNOY   FAISS    HNSW
Jumlah Data/Dimensi Metric                           
1000/2D             euclidean  0.0186  0.0184  0.0125
                    cosine     0.0160  0.0188  0.0107
1000/5D             euclidean  0.0157  0.0177  0.0156
                    cosine     0.0168  0.0189  0.0153
1000000/2D          euclidean  0.0459  0.1598  0.0270
                    cosine     0.0384  0.1716  0.0501
1000000/5D          euclidean  0.0466  0.1562  0.0438
                    cosine     0.0474  0.1671  0.0401

In [7]:
print('=== Recall@10 ===')
display(make_table('Recall@10').round(4))

=== Recall@10 ===


Algoritma                      ANNOY  FAISS   HNSW
Jumlah Data/Dimensi Metric                        
1000/2D             euclidean  0.994  1.000  1.000
                    cosine     0.996  1.000  0.999
1000/5D             euclidean  0.960  1.000  1.000
                    cosine     0.982  1.000  1.000
1000000/2D          euclidean  0.995  1.000  1.000
                    cosine     0.079  0.965  0.241
1000000/5D          euclidean  0.952  1.000  1.000
                    cosine     0.979  1.000  0.999